# OpenPlaque — diameter-constrained left coronary ostium neck detector

This is a fresh source-volume experiment. It does **not** launch another long graph search and it does **not** attempt LAD/LCX tracking. Instead it skeletonizes contrast-filled anatomy in a small high-resolution aortic-root crop and keeps only medial skeleton segments whose local 3-D radius is coronary-sized. Broad chambers should therefore disappear before candidate generation.


## Step 1 — Mount Google Drive


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache reuse controls


In [ ]:
REUSE_SOURCE_CT = True
REUSE_ROOT_CROP = True
REUSE_RCA_CALIBRATION = True
REUSE_NECK_CANDIDATES = True
REUSE_NECK_QC = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install this branch


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch left-coronary-ostium-neck-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas psutil "pylibjpeg>=2.0" "pylibjpeg-libjpeg>=2.1"
import sys, os, gc, psutil
sys.path.insert(0, '/content/OpenPlaque/src')
def ram(label):
    p=psutil.Process(os.getpid())
    print(f'{label}: RSS {p.memory_info().rss/1024**3:.2f} GB')
ram('After install')


## Step 4 — Initialize workflow and inspect cache plan


In [ ]:
from openplaque.left_ostium_neck import LeftOstiumNeckWorkflow, ALGORITHM_VERSION
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'root_crop': REUSE_ROOT_CROP,
    'rca_calibration': REUSE_RCA_CALIBRATION,
    'neck_candidates': REUSE_NECK_CANDIDATES,
    'neck_qc': REUSE_NECK_QC,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = LeftOstiumNeckWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=reuse)
print('Algorithm:', ALGORITHM_VERSION)
display(wf.cache_status())


## Step 5 — Reuse disk-backed source CCTA and build the local aortic-root crop

The full CCTA stays disk-backed. Only the small root crop is materialized. The TotalSegmentator aorta mask is used only to define the wall and outside-aorta distance.


In [ ]:
wf.load_source_ct()
wf.build_root_crop()
gc.collect(); ram('After source CT + root skeleton')
print('Root crop shape:', wf.root_ct.shape)
print('Root crop origin z,y,x:', wf.crop_lo)


## Step 6 — Recalibrate coronary size from the accepted RCA


In [ ]:
rca = wf.calibrate_rca()
display(rca)
display(wf.rca_qc)


## Step 7 — Find diameter-constrained ostium-neck skeleton components

Candidate generation now happens on the medial skeleton of contrast-filled structures outside the aorta. Skeleton voxels with local radius larger than the RCA-calibrated coronary range are discarded before any path is proposed. Candidates must touch the aortic wall, extend at least 5 mm away from it, be separated from the known RCA ostium, and lie on a substantially different aortic circumference angle.


In [ ]:
candidates = wf.find_neck_candidates()
display(candidates.head(30))
gc.collect(); ram('After neck candidate generation')


## Step 8 — Serial source-volume lumen QC

The candidate generator is independent of the prior ray/long-graph family, but the successful RCA-based orthogonal-plane acceptance gate is retained.


In [ ]:
summary = wf.qc_necks()
print('Selected neck summary:')
display(summary)
display(wf.best_qc)
gc.collect(); ram('After neck QC')


## Step 9 — QC figures

Inspect the local MIPs, the top candidate cross-sections, and especially the direct RCA-vs-neck comparison. A PASS is not accepted anatomically until these images are convincing.


In [ ]:
figs = wf.plot_qc()
for f in figs:
    print('Saved:', f)
gc.collect(); ram('After figures')


## Step 10 — Package report-back ZIP


In [ ]:
zip_path = wf.package()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LEFT_OSTIUM_NECK_REPORT_BACK.zip')


## Step 11 — Report back

After the ZIP is written, return to ChatGPT and say **Retrieve and analyze**. Do not start LAD/LCX tracking unless this ostium-neck detector is anatomically accepted.
